In [1]:
from langchain_openai import ChatOpenAI
from langchain.messages import HumanMessage,SystemMessage
from langchain_core.prompts import ChatPromptTemplate,PromptTemplate,MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import DirectoryLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma, FAISS
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langgraph.graph import StateGraph,START, END,MessagesState
from typing import TypedDict,Annotated,Literal
import os 
import sys
sys.path.insert(1, r'D:\Notebooks\LLM\env')
#sys.path.insert(2, r'D:\Notebooks\LLM\langchain_document_loader')
from enviorment import load_env
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_core.tools import tool
#from pydirectoryloader import rag_function
import os 
load_env()
from langchain_community.tools.tavily_search import TavilySearchResults

from langchain_community.document_loaders import WebBaseLoader


USER_AGENT environment variable not set, consider setting it to identify your requests.


In [3]:
from langchain_core.prompts import HumanMessagePromptTemplate

In [2]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import time
def get_ruls(start_url,max_urls):

# A set to store unique URLs we have visited
    visited_urls = set()
    # A queue (list) to store URLs we still need to visit
    urls_to_visit = []
    # The set to store the final list of internal links
    internal_urls = set()

    urls_to_visit.append(start_url)
    domain_name = urlparse(start_url).netloc

    while urls_to_visit and len(internal_urls) < max_urls:
        current_url = urls_to_visit.pop(0)

        if current_url in visited_urls:
            continue

        print(f"Fetching: {current_url}")
        visited_urls.add(current_url)

        try:
            response = requests.get(current_url, timeout=5)
            response.raise_for_status() # Check for bad status codes
        except requests.exceptions.RequestException as e:
            print(f"Failed to retrieve {current_url}: {e}")
            continue

        soup = BeautifulSoup(response.content, 'html.parser')

        for anchor_tag in soup.find_all('a', href=True):
            href = anchor_tag['href']
            # Join relative URLs with the base URL to make them absolute
            full_url = urljoin(current_url, href)
            # Parse the URL to verify its domain
            parsed_url = urlparse(full_url)

            # Ensure the link is internal and valid
            if parsed_url.netloc == domain_name and parsed_url.scheme in ['http', 'https']:
                # Clean the URL to ignore fragments (e.g., #section)
                clean_url = urljoin(full_url, urlparse(full_url).path)
                
                if clean_url not in internal_urls:
                    internal_urls.add(clean_url)
                    urls_to_visit.append(clean_url)
                    #print(f"  Found new URL: {clean_url}")
        
        # Be respectful: add a small delay between requests
        #time.sleep(1)
        return list(internal_urls)

In [3]:
url='https://fisat.ac.in/'
l=get_ruls(url,50)

Fetching: https://fisat.ac.in/


In [4]:
len(l)

98

In [5]:
model=ChatOpenAI(model='gpt-4o',temperature=0, max_completion_tokens=2000)

In [6]:
from pydantic import BaseModel,Field

In [8]:
class output(BaseModel):
    urls :list[str]=Field(description='contains the list of the relevant url')
model_stuctured=model.with_structured_output(output)
prompt =ChatPromptTemplate.from_messages([('system','you are URL selector who helps getting the relevant url from the list for a given question '),
                                          ('human','for a given question {question} , get me the relevant top 10 urls  from the URLS list {URL} which can help to answer the question ')])

In [62]:
prompt

ChatPromptTemplate(input_variables=['URL', 'question'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='you are URL selector who helps getting the relevant url from the list for a given question '), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['URL', 'question'], input_types={}, partial_variables={}, template='for a given question {question} , get me the relevant top 10 urls  from the URLS list {URL} which can help to answer the question '), additional_kwargs={})])

In [11]:
l=get_ruls('https://fisat.ac.in/',100)
question='tell me placements in FISAT in  year 2023'

chain=prompt| model_stuctured
result=chain.invoke({'question':question,"URL":l})

Fetching: https://fisat.ac.in/


In [12]:
urls=result.urls
urls

['https://fisat.ac.in/placements/',
 'https://fisat.ac.in/news/placement-diaries/',
 'https://fisat.ac.in/news/fisat-nsdc-global-giants-a-future-ready-you/',
 'https://fisat.ac.in/news/fisat-bags-1st-position-in-kerala/',
 'https://fisat.ac.in/news/reaccredited-by-the-naac-with-an-a-grade-in-the-2nd-cycle-with-a-score-of-3-45-cgpa/',
 'https://fisat.ac.in/news/all-b-tech-programs-now-accredited-by-national-board-of-accreditation-nba/',
 'https://fisat.ac.in/news/centre-for-future-skills-cfs-fisat/',
 'https://fisat.ac.in/news/technical-talk-on-renewable-energy-pathways-to-a-sustainable-future-and-the-role-of-anert/',
 'https://fisat.ac.in/news/fistaa-alumni-day-2025/',
 'https://fisat.ac.in/news/tender-notice-setting-up-of-aicte-idea-lab/']

In [13]:
#eval(urls)
urls

['https://fisat.ac.in/placements/',
 'https://fisat.ac.in/news/placement-diaries/',
 'https://fisat.ac.in/news/fisat-nsdc-global-giants-a-future-ready-you/',
 'https://fisat.ac.in/news/fisat-bags-1st-position-in-kerala/',
 'https://fisat.ac.in/news/reaccredited-by-the-naac-with-an-a-grade-in-the-2nd-cycle-with-a-score-of-3-45-cgpa/',
 'https://fisat.ac.in/news/all-b-tech-programs-now-accredited-by-national-board-of-accreditation-nba/',
 'https://fisat.ac.in/news/centre-for-future-skills-cfs-fisat/',
 'https://fisat.ac.in/news/technical-talk-on-renewable-energy-pathways-to-a-sustainable-future-and-the-role-of-anert/',
 'https://fisat.ac.in/news/fistaa-alumni-day-2025/',
 'https://fisat.ac.in/news/tender-notice-setting-up-of-aicte-idea-lab/']

In [8]:
urls=l[:4]

In [9]:
loader = WebBaseLoader(urls)
docs_lazy = loader.lazy_load()

In [44]:
docs_lazy

<generator object WebBaseLoader.lazy_load at 0x00000216F3CC8140>

In [10]:
docs=[]
for doc in docs_lazy:
    docs.append(doc)
#print(docs[0].page_content[:100])
#print(docs[0].metadata) 

In [11]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)
db=FAISS.from_documents(documents, OpenAIEmbeddings())
#db.save_local('vectorstore_index')
retriever=db.as_retriever(search_type='similarity',search_kwargs={"k":3})

In [26]:
len(documents[13].page_content)


697

In [17]:
def retriver_questions(result):
    #result=retriever.invoke(question)
    context_text= "\n".join([doc.page_content for doc in result])
    return context_text

In [18]:
model=ChatOpenAI()
#from langchain.schema.runnable import RunnableSequence,
from langchain_core.runnables import RunnableSequence, RunnableParallel,RunnablePassthrough



In [19]:
parallel_chain=RunnableParallel({'question':RunnablePassthrough(),'context':retriever|retriver_questions})

In [21]:
parallel_chain

{
  question: RunnablePassthrough(),
  context: VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000233D6DFFB60>, search_kwargs={'k': 3})
           | RunnableLambda(retriver_questions)
}

In [20]:
question='tell me placements in FISAT in  year 2023'
parallel_chain.invoke(question)

{'question': 'tell me placements in FISAT in  year 2023',
 'context': "2023\n\n\n\n\n\n                                                    Placement for 2023\n2025\n\n\n\n\n\n                                                    Placement for 2025                                                \n\n\n\nIt is truly inspiring to see that the FISAT Class of 2025 has elevated the placement statistics to a new benchmark, even amidst challenging industrial conditions and the widespread influence of AI. A standout highlight is the growing interest shown by leading core companies in FISATians, reflected in the exceptional performance of Civil, Mechanical, and Electrical Engineering students, who have secured an average package of ₹5.2 LPA, closely aligned with the ₹5.7 LPA average of their peers in Computer Science and Electronics. What makes this year even more remarkable is the exceptional performance of the Computer Science and Design students, who achieved the highest placement percentage of 

In [33]:
#human_message_template = HumanMessagePromptTemplate.from_template("Answer this quesiton {question} from the provided context only and here is the context {context} and if you dont the please say dont know")
#human_message_template = systemb.from_template("Answer this quesiton {question} from the provided context only and here is the context {context} and if you dont the please say dont know")
#human = HumanMessage("Answer this quesiton {question} from the provided context only and here is the context {context} and if you dont the please say dont know")

#system=SystemMessage(content='You are a college professor of BITS pilani office who helps students to resolve queries')
prompt=ChatPromptTemplate.from_messages([('system','You are a college professor of BITS pilani office who helps students to resolve queries'),('human','Answer this quesiton {question} from the provided context only and here is the context {context} and if you dont the please say dont know'), MessagesPlaceholder(variable_name="history")])

In [34]:
prompt

ChatPromptTemplate(input_variables=['context', 'history', 'question'], input_types={'history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotat

In [24]:
parser=StrOutputParser()

In [25]:
final_chain=parallel_chain | prompt|model|parser

In [26]:
question ='tell me placements in FISAT in  year 2023'
parallel_chain.invoke(question)

{'question': 'tell me placements in FISAT in  year 2023',
 'context': "2023\n\n\n\n\n\n                                                    Placement for 2023\n2025\n\n\n\n\n\n                                                    Placement for 2025                                                \n\n\n\nIt is truly inspiring to see that the FISAT Class of 2025 has elevated the placement statistics to a new benchmark, even amidst challenging industrial conditions and the widespread influence of AI. A standout highlight is the growing interest shown by leading core companies in FISATians, reflected in the exceptional performance of Civil, Mechanical, and Electrical Engineering students, who have secured an average package of ₹5.2 LPA, closely aligned with the ₹5.7 LPA average of their peers in Computer Science and Electronics. What makes this year even more remarkable is the exceptional performance of the Computer Science and Design students, who achieved the highest placement percentage of 

In [31]:
question ='tell me placements in FISAT for year 2025'
#final_chain.invoke(question)
final_chain.invoke(question)

'Based on the provided context, the information regarding placements at FISAT for the year 2025 is as follows:\n\n- The Class of 2025 at FISAT has achieved impressive placement statistics, reaching a new benchmark despite challenging industrial conditions and the impact of AI.\n- Notable interest has been shown by leading core companies in FISATians, particularly in Civil, Mechanical, and Electrical Engineering students, who have secured an average package of ₹5.2 LPA. This average is comparable to the ₹5.7 LPA average of their peers in Computer Science and Electronics.\n- Computer Science and Design students have excelled in placements, achieving the highest placement percentage of 91.6% in the class of 2025.\n- The highest package offered to students in 2025 is ₹11 LPA.\n\nIf you are looking for more specific details beyond what is provided in the context, I am unable to provide that information as it is not mentioned.'

In [38]:
from langchain_community.document_loaders.firecrawl import FireCrawlLoader

In [39]:
urls

['https://fisat.ac.in/placements/',
 'https://fisat.ac.in/news/placement-diaries/',
 'https://fisat.ac.in/news/fisat-nsdc-global-giants-a-future-ready-you/',
 'https://fisat.ac.in/news/fisat-bags-1st-position-in-kerala/',
 'https://fisat.ac.in/news/reaccredited-by-the-naac-with-an-a-grade-in-the-2nd-cycle-with-a-score-of-3-45-cgpa/',
 'https://fisat.ac.in/news/all-b-tech-programs-now-accredited-by-national-board-of-accreditation-nba/',
 'https://fisat.ac.in/news/centre-for-future-skills-cfs-fisat/',
 'https://fisat.ac.in/news/technical-talk-on-renewable-energy-pathways-to-a-sustainable-future-and-the-role-of-anert/',
 'https://fisat.ac.in/news/fistaa-alumni-day-2025/',
 'https://fisat.ac.in/news/tender-notice-setting-up-of-aicte-idea-lab/']

In [ ]:
urls[:2]

['https://fisat.ac.in/placements/',
 'https://fisat.ac.in/news/placement-diaries/',
 'https://fisat.ac.in/news/fisat-nsdc-global-giants-a-future-ready-you/',
 'https://fisat.ac.in/news/fisat-bags-1st-position-in-kerala/',
 'https://fisat.ac.in/news/reaccredited-by-the-naac-with-an-a-grade-in-the-2nd-cycle-with-a-score-of-3-45-cgpa/']

In [44]:
all_docs=[]
for url in urls[:2]:
    loader = FireCrawlLoader(
        api_key="fc-418ca318825e406fb656fd3c591812d8",
        url=url,
        mode="scrape"  # only scrape single page
    )
    docs = loader.load()
    all_docs.extend(docs)

In [45]:
all_docs

[Document(metadata={'title': 'Placements |  FISAT | Federal Institute of Science And Technology', 'description': None, 'url': 'https://fisat.ac.in/placements/', 'language': 'en', 'keywords': None, 'robots': 'max-image-preview:large', 'og_title': None, 'og_description': None, 'og_url': None, 'og_image': None, 'og_audio': None, 'og_determiner': None, 'og_locale': None, 'og_locale_alternate': None, 'og_site_name': None, 'og_video': None, 'favicon': 'https://fisat.ac.in/wp-content/uploads/2022/06/cropped-FISAT-32x32.png', 'dc_terms_created': None, 'dc_date_created': None, 'dc_date': None, 'dc_terms_type': None, 'dc_type': None, 'dc_terms_audience': None, 'dc_terms_subject': None, 'dc_subject': None, 'dc_description': None, 'dc_terms_keywords': None, 'modified_time': None, 'published_time': None, 'article_tag': None, 'article_section': None, 'source_url': 'https://fisat.ac.in/placements/', 'status_code': 200, 'scrape_id': 'ddde9763-fc01-44b8-b76b-d6b4117702ca', 'num_pages': None, 'content_t

In [32]:
from langchain_community.document_loaders import WikipediaLoader

# Example: Load documents for "Hunter x Hunter"
docs = WikipediaLoader(query="India", load_max_docs=10).load()


In [38]:
docs[0].metadata

{'title': 'India',
 'summary': "India, officially the Republic of India, is a country in South Asia.  It is the seventh-largest country by area; the most populous country since 2023; and, since its independence in 1947, the world's most populous democracy. Bounded by the Indian Ocean on the south, the Arabian Sea on the southwest, and the Bay of Bengal on the southeast, it shares land borders with Pakistan to the west; China, Nepal, and Bhutan to the north; and Bangladesh and Myanmar to the east. In the Indian Ocean, India is near Sri Lanka and the Maldives; its Andaman and Nicobar Islands share a maritime border with Myanmar, Thailand, and Indonesia.\nModern humans arrived on the Indian subcontinent from Africa no later than 55,000 years ago. Their long occupation, predominantly in isolation as hunter-gatherers, has made the region highly diverse. Settled life emerged on the subcontinent in the western margins of the Indus river basin 9,000 years ago, evolving gradually into the Indus